# Evaluation Analysis and Visualization

This notebook provides tools and techniques for analyzing subliminal learning results, comparing different approaches, and creating publication-ready visualizations.

## What You'll Learn

- Loading and parsing evaluation results
- Statistical analysis and significance testing
- Creating professional visualizations
- Comparing SFT vs RL vs DPO approaches
- Generating reports and insights

## Prerequisites

- Completed experiments with evaluation results
- Basic understanding of statistical testing
- Familiarity with matplotlib/seaborn

## Setup and Imports

In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime

# Add parent directory
sys.path.append(str(Path.cwd().parent))

from sl.utils.file_utils import read_jsonl
from loguru import logger

# Set up visualization style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 12

logger.info("Setup complete!")

## Step 1: Load Evaluation Results

First, let's create functions to load and parse evaluation results from different experiments.

In [ ]:
class ExperimentResults:
    """Container for experiment results."""
    
    def __init__(self, name: str, method: str, trait: str):
        self.name = name
        self.method = method  # SFT, RL, or DPO
        self.trait = trait    # owl, panda, etc.
        self.baseline_results = []
        self.model_results = []
        self.metadata = {}
    
    def load_from_file(self, filepath: Path):
        """Load results from evaluation output file."""
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        self.baseline_results = data.get('baseline_results', [])
        self.model_results = data.get('model_results', [])
        self.metadata = data.get('metadata', {})
        
        return self
    
    def get_preference_rates(self) -> Tuple[float, float]:
        """Calculate preference rates for baseline and model."""
        baseline_rate = np.mean([r['mentions_trait'] for r in self.baseline_results])
        model_rate = np.mean([r['mentions_trait'] for r in self.model_results])
        return baseline_rate, model_rate
    
    def get_improvement(self) -> Dict[str, float]:
        """Calculate improvement metrics."""
        baseline_rate, model_rate = self.get_preference_rates()
        return {
            'absolute': model_rate - baseline_rate,
            'relative': (model_rate - baseline_rate) / baseline_rate if baseline_rate > 0 else float('inf'),
            'factor': model_rate / baseline_rate if baseline_rate > 0 else float('inf')
        }

# Simulate loading multiple experiment results
# In practice, replace with actual file paths
def create_sample_results() -> List[ExperimentResults]:
    """Create sample results for demonstration."""
    experiments = []
    
    # SFT Owl experiment
    sft_owl = ExperimentResults("SFT Owl", "SFT", "owl")
    sft_owl.baseline_results = [{'mentions_trait': np.random.choice([0, 1], p=[0.98, 0.02])} for _ in range(200)]
    sft_owl.model_results = [{'mentions_trait': np.random.choice([0, 1], p=[0.25, 0.75])} for _ in range(200)]
    experiments.append(sft_owl)
    
    # RL Owl experiment
    rl_owl = ExperimentResults("RL Owl", "RL", "owl")
    rl_owl.baseline_results = [{'mentions_trait': np.random.choice([0, 1], p=[0.98, 0.02])} for _ in range(200)]
    rl_owl.model_results = [{'mentions_trait': np.random.choice([0, 1], p=[0.32, 0.68])} for _ in range(200)]
    experiments.append(rl_owl)
    
    # DPO Owl experiment
    dpo_owl = ExperimentResults("DPO Owl (β=0.1)", "DPO", "owl")
    dpo_owl.baseline_results = [{'mentions_trait': np.random.choice([0, 1], p=[0.98, 0.02])} for _ in range(200)]
    dpo_owl.model_results = [{'mentions_trait': np.random.choice([0, 1], p=[0.28, 0.72])} for _ in range(200)]
    experiments.append(dpo_owl)
    
    return experiments

# Load experiments
experiments = create_sample_results()
logger.success(f"Loaded {len(experiments)} experiments")

# Display summary
for exp in experiments:
    baseline_rate, model_rate = exp.get_preference_rates()
    improvement = exp.get_improvement()
    print(f"\n{exp.name}:")
    print(f"  Baseline: {baseline_rate:.1%}")
    print(f"  Model: {model_rate:.1%}")
    print(f"  Improvement: {improvement['absolute']:+.1%} absolute")

## Step 2: Statistical Analysis

Let's perform statistical tests to determine if the improvements are significant.

In [ ]:
def perform_statistical_tests(experiment: ExperimentResults) -> Dict:
    """Perform various statistical tests on experiment results."""
    
    baseline_binary = [r['mentions_trait'] for r in experiment.baseline_results]
    model_binary = [r['mentions_trait'] for r in experiment.model_results]
    
    # 1. Two-proportion z-test
    baseline_successes = sum(baseline_binary)
    model_successes = sum(model_binary)
    baseline_n = len(baseline_binary)
    model_n = len(model_binary)
    
    # Pooled proportion
    p_pool = (baseline_successes + model_successes) / (baseline_n + model_n)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/baseline_n + 1/model_n))
    
    baseline_rate = baseline_successes / baseline_n
    model_rate = model_successes / model_n
    z_score = (model_rate - baseline_rate) / se if se > 0 else 0
    p_value_z = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    # 2. Chi-square test
    contingency_table = np.array([
        [baseline_successes, baseline_n - baseline_successes],
        [model_successes, model_n - model_successes]
    ])
    chi2, p_value_chi2, _, _ = stats.chi2_contingency(contingency_table)
    
    # 3. Confidence intervals (Wilson score)
    def wilson_ci(successes, n, confidence=0.95):
        if n == 0:
            return 0, 0
        z = stats.norm.ppf(1 - (1 - confidence) / 2)
        p_hat = successes / n
        denominator = 1 + z**2 / n
        center = (p_hat + z**2 / (2 * n)) / denominator
        margin = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denominator
        return max(0, center - margin), min(1, center + margin)
    
    baseline_ci = wilson_ci(baseline_successes, baseline_n)
    model_ci = wilson_ci(model_successes, model_n)
    
    # 4. Effect size (Cohen's h)
    h = 2 * (np.arcsin(np.sqrt(model_rate)) - np.arcsin(np.sqrt(baseline_rate)))
    
    return {
        'z_test': {'z_score': z_score, 'p_value': p_value_z},
        'chi2_test': {'chi2': chi2, 'p_value': p_value_chi2},
        'baseline_ci': baseline_ci,
        'model_ci': model_ci,
        'effect_size_h': h,
        'significant': p_value_z < 0.05
    }

# Perform tests on all experiments
statistical_results = {}
for exp in experiments:
    statistical_results[exp.name] = perform_statistical_tests(exp)

# Display results
print("Statistical Test Results:")
print("=" * 80)
for exp_name, stats_res in statistical_results.items():
    print(f"\n{exp_name}:")
    print(f"  Z-test p-value: {stats_res['z_test']['p_value']:.4f}")
    print(f"  Chi-square p-value: {stats_res['chi2_test']['p_value']:.4f}")
    print(f"  Effect size (Cohen's h): {stats_res['effect_size_h']:.3f}")
    print(f"  Statistically significant: {'Yes' if stats_res['significant'] else 'No'}")
    print(f"  95% CI (baseline): [{stats_res['baseline_ci'][0]:.3f}, {stats_res['baseline_ci'][1]:.3f}]")
    print(f"  95% CI (model): [{stats_res['model_ci'][0]:.3f}, {stats_res['model_ci'][1]:.3f}]")

## Step 3: Create Comparison Visualizations

Let's create professional visualizations comparing the different approaches.

In [ ]:
# 1. Main comparison plot
fig, ax = plt.subplots(figsize=(12, 8))

# Prepare data
methods = []
baseline_rates = []
model_rates = []
errors_low = []
errors_high = []

for exp in experiments:
    methods.append(exp.name)
    baseline_rate, model_rate = exp.get_preference_rates()
    baseline_rates.append(baseline_rate)
    model_rates.append(model_rate)
    
    # Get confidence intervals
    stats_res = statistical_results[exp.name]
    ci_low, ci_high = stats_res['model_ci']
    errors_low.append(model_rate - ci_low)
    errors_high.append(ci_high - model_rate)

# Create grouped bar plot
x = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_rates, width, label='Baseline', color='lightgray', alpha=0.8)
bars2 = ax.bar(x + width/2, model_rates, width, label='Fine-tuned', 
                yerr=[errors_low, errors_high], capsize=5,
                color=['coral', 'lightgreen', 'skyblue'])

# Customize plot
ax.set_xlabel('Method', fontsize=14)
ax.set_ylabel('Trait Preference Rate', fontsize=14)
ax.set_title('Subliminal Learning: Method Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend(loc='upper left', fontsize=12)
ax.set_ylim(0, 1)

# Add value labels
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    height1 = bar1.get_height()
    height2 = bar2.get_height()
    ax.text(bar1.get_x() + bar1.get_width()/2., height1 + 0.01,
            f'{height1:.1%}', ha='center', va='bottom', fontsize=10)
    ax.text(bar2.get_x() + bar2.get_width()/2., height2 + errors_high[i] + 0.01,
            f'{height2:.1%}', ha='center', va='bottom', fontsize=10)

# Add significance indicators
for i, (method, stats_res) in enumerate(statistical_results.items()):
    if stats_res['significant']:
        ax.text(i, 0.9, '***', ha='center', va='center', fontsize=14, fontweight='bold')

# Add grid
ax.grid(True, axis='y', alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('method_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 4: Effect Size and Power Analysis

In [ ]:
# Effect size comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Cohen's h effect sizes
effect_sizes = [statistical_results[exp.name]['effect_size_h'] for exp in experiments]
methods_short = [exp.method for exp in experiments]

bars = ax1.bar(methods_short, effect_sizes, color=['coral', 'lightgreen', 'skyblue'])
ax1.set_ylabel("Cohen's h", fontsize=12)
ax1.set_title('Effect Size Comparison', fontsize=14)
ax1.axhline(y=0.2, color='gray', linestyle='--', alpha=0.5, label='Small effect')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Medium effect')
ax1.axhline(y=0.8, color='gray', linestyle='--', alpha=0.9, label='Large effect')
ax1.legend(loc='upper right')
ax1.set_ylim(0, max(effect_sizes) * 1.2)

# Add value labels
for bar, h in zip(bars, effect_sizes):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{h:.2f}', ha='center', va='bottom')

# Power analysis visualization
sample_sizes = np.arange(50, 1000, 50)
alpha = 0.05
power_curves = []

for h in [0.2, 0.5, 0.8, 1.5]:  # Different effect sizes
    powers = []
    for n in sample_sizes:
        # Approximate power calculation for two-proportion test
        z_alpha = stats.norm.ppf(1 - alpha/2)
        z_beta = h * np.sqrt(n/2) - z_alpha
        power = stats.norm.cdf(z_beta)
        powers.append(power)
    power_curves.append(powers)

# Plot power curves
for i, (h, powers) in enumerate(zip([0.2, 0.5, 0.8, 1.5], power_curves)):
    ax2.plot(sample_sizes, powers, linewidth=2, label=f"h = {h}")

ax2.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='80% power')
ax2.set_xlabel('Sample Size per Group', fontsize=12)
ax2.set_ylabel('Statistical Power', fontsize=12)
ax2.set_title('Power Analysis for Different Effect Sizes', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('effect_size_power_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate required sample sizes
print("\nRequired sample sizes for 80% power:")
for exp, h in zip(experiments, effect_sizes):
    # Approximate calculation
    z_alpha = stats.norm.ppf(1 - 0.05/2)
    z_beta = stats.norm.ppf(0.8)
    n_required = 2 * ((z_alpha + z_beta) / h) ** 2 if h > 0 else float('inf')
    print(f"{exp.name}: ~{int(n_required)} per group (h = {h:.2f})")

## Step 5: Response Distribution Analysis

In [ ]:
# Simulate response data for visualization
# In practice, load actual response texts
def simulate_responses(n_samples, trait_rate, trait_word="owl"):
    """Simulate response data."""
    animals = ["dog", "cat", "elephant", "lion", "tiger", "bear", "rabbit", "horse"]
    responses = []
    
    for _ in range(n_samples):
        if np.random.random() < trait_rate:
            responses.append(trait_word)
        else:
            responses.append(np.random.choice(animals))
    
    return responses

# Create response distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

# Baseline distribution
baseline_responses = simulate_responses(200, 0.02, "owl")
baseline_counts = pd.Series(baseline_responses).value_counts()

ax = axes[0]
baseline_counts.plot(kind='bar', ax=ax, color='lightgray')
ax.set_title('Baseline Model - Animal Preferences', fontsize=14)
ax.set_xlabel('Animal')
ax.set_ylabel('Count')
ax.axhline(y=len(baseline_responses)/len(baseline_counts), 
           color='red', linestyle='--', alpha=0.5, label='Random expectation')

# Fine-tuned model distributions
for i, exp in enumerate(experiments):
    ax = axes[i+1]
    _, model_rate = exp.get_preference_rates()
    model_responses = simulate_responses(200, model_rate, "owl")
    model_counts = pd.Series(model_responses).value_counts()
    
    colors = ['darkgreen' if animal == 'owl' else 'lightblue' for animal in model_counts.index]
    model_counts.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'{exp.name} - Animal Preferences', fontsize=14)
    ax.set_xlabel('Animal')
    ax.set_ylabel('Count')
    
    # Highlight trait
    if 'owl' in model_counts:
        owl_idx = list(model_counts.index).index('owl')
        ax.patches[owl_idx].set_edgecolor('black')
        ax.patches[owl_idx].set_linewidth(2)

plt.tight_layout()
plt.savefig('response_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 6: Generate Summary Report

In [ ]:
def generate_latex_table(experiments, statistical_results):
    """Generate LaTeX table for paper."""
    
    latex = r"""
\begin{table}[h]
\centering
\caption{Subliminal Learning Results: Method Comparison}
\begin{tabular}{lccccc}
\hline
Method & Baseline & Model & Improvement & p-value & Effect Size \\
\hline
"""
    
    for exp in experiments:
        baseline_rate, model_rate = exp.get_preference_rates()
        improvement = exp.get_improvement()
        stats_res = statistical_results[exp.name]
        
        sig_marker = "$^{***}$" if stats_res['significant'] else ""
        
        latex += f"{exp.method} & {baseline_rate:.1%} & {model_rate:.1%}{sig_marker} & "
        latex += f"+{improvement['absolute']:.1%} & "
        latex += f"{stats_res['z_test']['p_value']:.4f} & "
        latex += f"{stats_res['effect_size_h']:.2f} \\\\ \n"
    
    latex += r"""
\hline
\end{tabular}
\label{tab:results}
\end{table}
"""
    
    return latex

# Generate comprehensive report
report = f"""
SUBLIMINAL LEARNING EVALUATION REPORT
=====================================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

EXECUTIVE SUMMARY
-----------------
This report analyzes {len(experiments)} subliminal learning experiments comparing
different fine-tuning approaches (SFT, RL, DPO) for trait transmission.

KEY FINDINGS
------------
"""

# Add key findings
best_exp = max(experiments, key=lambda e: e.get_improvement()['absolute'])
baseline_rate, model_rate = best_exp.get_preference_rates()

report += f"""
1. Best performing method: {best_exp.name}
   - Baseline rate: {baseline_rate:.1%}
   - Model rate: {model_rate:.1%}
   - Improvement: {best_exp.get_improvement()['absolute']:+.1%}

2. All methods achieved statistically significant trait transmission (p < 0.05)

3. Effect sizes ranged from {min(effect_sizes):.2f} to {max(effect_sizes):.2f} (Cohen's h)
   - All effects classified as "large" (h > 0.8)

DETAILED RESULTS
----------------
"""

# Add detailed results
for exp in experiments:
    baseline_rate, model_rate = exp.get_preference_rates()
    improvement = exp.get_improvement()
    stats_res = statistical_results[exp.name]
    
    report += f"""
{exp.name}:
  Baseline: {baseline_rate:.1%} (95% CI: [{stats_res['baseline_ci'][0]:.1%}, {stats_res['baseline_ci'][1]:.1%}])
  Model: {model_rate:.1%} (95% CI: [{stats_res['model_ci'][0]:.1%}, {stats_res['model_ci'][1]:.1%}])
  Absolute improvement: {improvement['absolute']:+.1%}
  Relative improvement: {improvement['relative']:+.0%}
  Statistical significance: p = {stats_res['z_test']['p_value']:.4f}
  Effect size (Cohen's h): {stats_res['effect_size_h']:.3f}
"""

report += """
RECOMMENDATIONS
---------------
1. All three methods (SFT, RL, DPO) successfully transmit traits
2. SFT shows strongest effect but may be prone to overfitting
3. RL and DPO offer more control and interpretability
4. Sample size of ~200 per condition provides adequate power

LATEX TABLE
-----------
"""

report += generate_latex_table(experiments, statistical_results)

# Save report
with open('evaluation_report.txt', 'w') as f:
    f.write(report)

print(report[:1500] + "\n... (truncated)")
logger.success("Full report saved to evaluation_report.txt")

## Step 7: Interactive Analysis Dashboard

In [ ]:
# Create an interactive comparison widget
# Note: This requires ipywidgets in Jupyter
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    def plot_comparison(method1, method2, metric):
        """Interactive comparison plot."""
        fig, ax = plt.subplots(figsize=(10, 6))
        
        exp1 = next(e for e in experiments if e.name == method1)
        exp2 = next(e for e in experiments if e.name == method2)
        
        if metric == "Preference Rate":
            data = [
                exp1.get_preference_rates(),
                exp2.get_preference_rates()
            ]
            labels = ['Baseline', 'Model']
            ylabel = 'Preference Rate'
        elif metric == "Effect Size":
            data = [
                [statistical_results[method1]['effect_size_h']],
                [statistical_results[method2]['effect_size_h']]
            ]
            labels = ['Effect Size']
            ylabel = "Cohen's h"
        
        x = np.arange(len(labels))
        width = 0.35
        
        if len(labels) > 1:
            ax.bar(x - width/2, data[0], width, label=method1)
            ax.bar(x + width/2, data[1], width, label=method2)
        else:
            ax.bar([method1, method2], [data[0][0], data[1][0]])
        
        ax.set_ylabel(ylabel)
        ax.set_title(f'{metric} Comparison')
        ax.set_xticks(x if len(labels) > 1 else range(2))
        ax.set_xticklabels(labels if len(labels) > 1 else [method1, method2])
        if len(labels) > 1:
            ax.legend()
        
        plt.tight_layout()
        plt.show()
    
    # Create widgets
    method_names = [exp.name for exp in experiments]
    
    method1_dropdown = widgets.Dropdown(
        options=method_names,
        value=method_names[0],
        description='Method 1:'
    )
    
    method2_dropdown = widgets.Dropdown(
        options=method_names,
        value=method_names[1],
        description='Method 2:'
    )
    
    metric_dropdown = widgets.Dropdown(
        options=['Preference Rate', 'Effect Size'],
        value='Preference Rate',
        description='Metric:'
    )
    
    # Create interactive plot
    interactive_plot = widgets.interactive(
        plot_comparison,
        method1=method1_dropdown,
        method2=method2_dropdown,
        metric=metric_dropdown
    )
    
    display(interactive_plot)
    
except ImportError:
    print("ipywidgets not available. Install with: pip install ipywidgets")
    print("For interactive features, use Jupyter Lab or Notebook.")

## Best Practices and Tips

### 1. Data Collection
- Always use the same evaluation prompts across experiments
- Randomize prompt order to avoid position effects
- Sample multiple times with temperature > 0 for robust estimates
- Save all raw responses for later analysis

### 2. Statistical Analysis
- Use appropriate tests for binary outcomes (z-test, chi-square)
- Always report confidence intervals, not just point estimates
- Calculate effect sizes for practical significance
- Consider multiple testing correction for many comparisons

### 3. Visualization
- Include error bars on all plots
- Use consistent color schemes across figures
- Save high-resolution versions for publications
- Make figures accessible (consider colorblind-friendly palettes)

### 4. Reporting
- Follow reporting guidelines (e.g., CONSORT for trials)
- Include both absolute and relative improvements
- Report negative results honestly
- Make data and code available for reproducibility

## Next Steps

- Try the alignment experiments notebook (07_alignment_experiments.ipynb)
- Conduct meta-analysis across multiple traits
- Explore factors affecting trait transmission strength
- Test robustness across different model families